# 用一次真实聊天预测 76 aa 泛素

在免费 Colab 选择「运行时 → 更改运行时类型 → T4 GPU」。免费 GPU 不可用时暂停。左栏钥匙图标 **Secrets** 中添加名称 **DEEPSEEK_API_KEY**，值填本人已有的 DeepSeek API key，并为本笔记本开启 **Notebook access**。只在 Secrets 中输入值。

依次运行 1–7。完整泛素、后端、构建好的前端和运行包已内置。1–3 沿用第五轮已实测的 T4 环境。4–5 准备 FastAPI 和浏览器。第 6 单元通过本虚拟机的聊天 API 发送一条消息，DeepSeek 使用已有对话额度。一个后端进程处理聊天、调用官方 CLI，并向同一台虚拟机上的 Chromium 提供 Mol* 所需的结构。第 7 单元把证据下载到电脑。

In [ ]:
# @title 1 · 检查免费 T4、CUDA、显存和磁盘
import base64, hashlib, io, json, os, platform, re, shutil, subprocess, sys, uuid, zipfile
from pathlib import Path

gpu_ok = install_ok = assets_ok = prediction_ok = False
run_id = None
root = Path('/content/protenix-round5')
if platform.system() != 'Linux' or platform.machine() != 'x86_64' or not shutil.which('nvidia-smi'):
    raise RuntimeError('未检测到 Colab NVIDIA GPU。请检查运行时类型和免费配额。')
query = 'name,driver_version,memory.total,memory.free,compute_cap'
smi = subprocess.check_output(['nvidia-smi', '--query-gpu=' + query, '--format=csv,noheader,nounits'], text=True)
name, driver, total, free, cc = [s.strip() for s in smi.splitlines()[0].split(',')]
banner = subprocess.check_output(['nvidia-smi'], text=True)
cuda = re.search(r'CUDA Version:\s*([0-9.]+)', banner)
disk = shutil.disk_usage('/content').free / 2**30
nvcc = subprocess.check_output(['nvcc', '--version'], text=True).splitlines()[-1] if shutil.which('nvcc') else '未安装'
hardware = dict(gpu=name, driver=driver, driver_cuda=cuda[1] if cuda else 'unknown', toolkit=nvcc,
                capability=cc, total_vram_mib=int(total), free_vram_mib=int(free), free_disk_gib=round(disk, 2))
print(json.dumps(hardware, ensure_ascii=False, indent=2))
if 'T4' not in name or cc != '7.5':
    raise RuntimeError('本次验收使用免费 T4。请检查分配的显卡。')
root.mkdir(parents=True, exist_ok=True)
(root / 'hardware.json').write_text(json.dumps(hardware, indent=2))
gpu_ok = True
print('GPU OK ·', name)

In [ ]:
# @title 2 · 安装锁定环境，实测 CUDA 运算并检查官方 CLI
if not gpu_ok:
    raise RuntimeError('请先完成第 1 单元')
install_ok = assets_ok = prediction_ok = False
support = root / 'support'
payload = base64.b64decode('''UEsDBBQAAAAIAAAAIQA6s6hp5QwAAHggAAAgAAAAYmFja2VuZC9hcHAvYWRhcHRlcnMvcHJvdGVuaXgucHmVWftT20gS/t1/xWxy
VZISo4Xs7lXiWl0dAW/WG8AUkK1LEU41lsZ4gl6nkWycHP/7fT2jp21gj0pAmlf39PPr1osXL6aJYJlMEhGydD6XgeQRO5WJZEcn
EyaTZRrwQqYJy0TOvqYzl10tBJvx4E4kIUvEEsMyztK8UKxI82DhvnjxYjAwQ4yrdRLIdDDP05gFaVKI+yKSs2oHU2WW5UIpMx/y
ggcRV0qoekEz1KwQhYxFZ1q/Dxn9/pYmoib8VaVJ/ZxWuzNeLDrEz/FaL8mbjUreJjxqLnAr4lgOzH6eZa4q8jIoyrxlcZIseSTD
aVlkZTEYnE6PxyfMY1aWp4VI5L0fQ5h+KOa8jAp/ue/+4u5bg/PDo4+HH8a08I27TyOX4/ExXg/2DwYv2RmPQYFDwji8FKxYZ/Se
4wnSP4pkcMfSjPQCPhI9eL4+nzB9FlsthIjcwfT8ajI9u8ShoQwKO05DEfkJTvY0k0NWKuHHinu/8UgJ81qIOIsg1e5YnvDOsgHr
/wTrIBLez0OmCpF5v+AvxxHCOxiykNj2rHn20xsLKsolLwr/TuSJiDxLG4u1dRxWxWW0uYqJhM8i4c9LhTvXzFWDxfynN2bIGQwG
2lzYeSX9cZ6nuX1RJmQg+sUZaZIvyPKjNZvLe1h+Vs4gUyZoAew0hLCDPMU5JFloMoCRsllaJiHP18bCB1ApCyLpywSKt/My8WU4
ghRyiED8pxRJIPSrw/b+wSKpCkM4FzCfhF1/t0gV1oiZnUNm1bsUBjGtDUgmRwsuE4x8b+bxUj9iVwCuCgwdPDzcPNx0+OL5rbIz
noNKIXI10lbQMHMNzm4MR7QQRnINiiKErK29PX0neqyt2Iy45FZmRVoWfihzekm16ZthJUSoLDKGvEP72qJx68a50QTnEPKdWJPl
VjY6aszA2LvHOpux9KaZJ2ZdRBFEH/saBC32ms4yFPVmx43Slchth8k5k0omquAQlZkcQotp5DABa+lsuXG6uiEatSDl3Ke4YeOu
IoA1rkc6cJCOBdQtEyNSGuvpt1nPfmwkhMdm2CXtY2CuReN/p98PtEJrAZoi37b0gu/9TQ++8TB/3wV3VsUo5AVXbffasyidjdhs
XQg1ZM9Yp45y7mUd2xoPOVoIBBryARkKuFCxZiKeiTCEy8zWeqLJGatcQltDSCSIylAmt1rNIMWjvZoiJRCl3ce4+rrVO9gFKa9i
BRdzc8FDHxziKH0ZNxTkmbZVFvO9t5bjuCqFFPRG22kOAp8QumcOdOcyCX2tY9vy9ZQrQ6tdDRNJ0qLaBH5b8lzVxPWkw37wKjHS
OnO61iHGoSMz1eqmVgV0O1v7tRtZo168y7mEGf5J7JlAZZntlbjnUuQdXu/X33r3gv37vEhjX0Hy7hHPi8S/JzfcGl3vHP1m3WwJ
IhKJDToO3ZEna5vG9t69e+fiP/sVN41SXthLh57f1eOk6DxdkT8Tj/S6pBeMOc9dOEjTHNaClKMY3ETh5ojJCbhk58fvsSG5FR0Z
NPm3MZWY3wm/GfYpU1c2oX/3bki3a5ZqjR7QRXvD1/s3ZuY5znU+pe0BRWhm4nBLzox6rHsw/nXZsSz3ayoT21xEm2rBZyXl39AH
xJAhzDbXVua4ADd+JAr4mE9uYIROUtaENMuNYz9nZHUiabnVpxmKzZn9YwKOy1RLDK9kT7Z1dEjG9apzVnU7bJCKnYFvLSTuikjE
MOvGa6yjDXd4RM5SKQonPMoWHOfkM+SgnlpVGdukQwSzHVKpTfo5qZSJjqEho3upioK4D0RWsB6CGHa2DYH/QnFfPU+BhuEhKwM0
gH5pf0vX0OwBRtuS5rVBLCyOjya/WbgKYU7s72aVxpiqqC/uCYP6WTjblaA6Yf7xfIVYB8Vu5zmzw0gBS9oJVyIyruNIJgi7WrU6
WCMYL4WtJd4ubYd/rJbBuOUSmC1tD+xbDg6koEOrQWguI2Go0OA++1VPIKMXNuJ/gXD2TVAseuvv7+/Tf+cZcdfWhBPxCKwbiI7s
TfitdN+NNVvptclQOsPaznAjtbfyN4dBRczro4LmfBeTll61ksWClrppBqu17jXyhbuDY8/iKpDS0maFrYLHo25YxLurc7A9ty7G
p4cXH9m7X/bZH9P3bBNAfEk6C6gWYBfjw5Pe6HddIjwwXZUYcPIl6cfhlmB7ER2MqeCJeaSt0ukBK4zUhrtEJNhhswW/rQ321as5
DwqlzZWiiLntS/ZRiKzB5AZZ4VUuETVdpjF9w0/I9BHGlYoFwhGCdon8CuKoWzW4h/9QFcGi9NZtdWD3NAU4vac5VhoBR5ZTKYj3
FFQBk10KAoO4L212wzLOlK1LsrIIvLqGdZN0ZddlrIsZB/afIp7FZOuVaDz9u5UNxslDPEPOqYQbSn6bpKqQgW3kU+FALUosZf/t
CBSFpsKZME6ztkbPtePPCL0C5mFhDHFYFFzrPQSEaDqdm0ldP3VXdJzRWEBbFqdpbNU0CGlAIXH3bIrk9Zg9s5KUJUuJm7EQuhZU
eMysoAy5KdraVyiZQgVfchmRYvUMdpuKElU7J0DhPMEZHfP/sFbmEVlDh5FFUWSd1zBdJcBNITOQISBUnSHxFzBPaC98mp16t9Vq
BGATtOci16DaHDL6i3IX9yIoKYRZXc8ke4D16H4NIxuizs+dsM1qkmPtolQcjdoSshP36hUiWZpis+vP2va0FY62wX/tzR7jKy6b
tpEbwKoLoLtyVq3Q3FO86TAGH3hFTA1ZsAq9Tv4CHx7+b7UYmh9VwG+9mlhLxT0e/3n26eSE+A9h3ruWnE/Ox0+eDPXv2nd5dTz9
dKUlg8SdiJWvMExNjau8FD3QMb3UOOIRGNHvcHSsN5Kb4KGAgVDlYNUq54hDGNBKpxFUZSJfIlx67PvDtno2gnWlVhTelq+vIboV
1WoBa4SNl8kdG9UKrW9v5KkTp/324N2bDVBWMWrrv6/NIc713sHff3r78+imt7S5RCfY0TaNFczkJiCdWWyPHX06PvT/nFxO3p+M
feh5cjS+HLFr7Ty0fxuR1sK51vEF6ooElXXWDYiTzjbJIIYTwBgRWnaV4HmwsPOZdap9H6FjxfNQ9ytHzP4Svv7i4pdDPRPN/lP0
q81gIUiTUGkOTEFmaF4f3GyjcCpL1BY3Zz5sM9YMwH/OgMfuEMGaV4LA7VusuHn5MnueTbfMKKXZ32Rmo7wjGE6XszQF/WTwNR5w
rh/C0hcWyMQ8sxEShxXH7m2elsiTjuN0aioqfDZtil7snuO850qM9SP8quX0JTvUcg+peaXb2zFH0b9KtFGwVZojRyiT/2WyEIRt
QpbJTLAFT8JIKLe1clpUt6/tc8PKSZrelVm3y9gIR7l3MoqyW7vmOqOWn2k2u5eTDx8nJyftPfsXDNI4LhMZkFSdv+KUktBNXmZ9
x9RhY/DMVnFPlyZUgwddaHr0a9hxM894F4GQWucNWNAKahzw6ZBVeTBW70pOPdxIJofs9M/mc4CN+PZN1DGz3/I9DDluno8qi+jn
r+pUlZZ5IKiB2RkGqqBADAPT4HegB6veWtWLtOH9c53KKL9tZdnvVsaDO8jSr86ycLzp9Ov+LiQ70pAabxEHXEoA73zqkmO86XW/
elV1RR8MC21SruoPzcVWgbeVh3f2ftvCz7jEH+nsEjqUwBEm2yrGkT2EWtSgulPYMHhVVCrgL7OYpJWLr4IwNmwIcd8UUMDlNEHd
tr7H9GH1ruays7P0eQJZ00+Dq+22If9oOTasjqBWJbXZvDetj9i72raOG99hvNdjXFKqvBuRY+NF5mlyfXdjGtsaFVq/T0/HFOPO
D69+p79Xp+fHkwt6Ojk8+6D/Hvsnk/cXhxef/XqRTk5m56PQYvvH2pXTmuP02boXrjlrGX7oXqeO2rSaqgltYm7rPS6MSVDv4DWd
QJ0ChSrstd56K1BXV1fAJAyV5p2dVzi/mF6Nzyb/8i+m0ysfEmmJdb0SSjo5/Dy+OJtenPpXn8/HXvdjwobr3Oym9Pnq9+nZ2fTT
5fjicnI19qyDTjBsfd3E2grubtx6yOgTQ+3LXayr0SVEXM+1J6NcCxZsI93++8vl62FD1L7e33vn3iCfqld/s5rxIe05/XRyNTmZ
nI072QC1LCdcpI9Gkq874Vqrhp7+ntHgOa3UzSBPVWwTlyg69CKVZ6j0+moV4R+8OojtaqE9hkO3JVN9Z9rxbeqJrGZR9kP2tXTl
sfSuGwpWhfuhlr5BPGkrm0n2McWbimJL320HqHOxl/qbONCKjiv0lSAR1PGEYhAKM5HvVW19qtT0t2P6shtw6ucBFehca+Kjzp/f
RJ62cVOXkWqzd9R+UbLGFxdWV2tmAy6DEg34SdOiKrYap680OprtbIg+ps3NxKx3mKS3qwc57Hyh3PEZcPA/UEsDBBQAAAAIAAAA
IQByFsKMyAMAAN4IAAAZAAAAYmFja2VuZC9hcHAvc3RydWN0dXJlcy5weZ1V247bNhB911dM3AdJqaK4TTfYGnEBJzF63TpYLPpi
GAJFjSxmJVIlqXidtv/eISVfd+sWFXZhi5zLmTMzx6PR6DdWi4JZBHxg3NZbsBVCvrVo6BuzwJmEHMGg/oRFAqVWDbD+Vb9QG4kF
fFQ5tMxW6Wg0CgLRtEpbqJipapHvXhu6D7yzs6QLGC4+uIud1RqbRgRBwGtmDPwoPzlwi862nY0IaIdzrZWOJwHQ05IN2RZYAmER
pcAia4s8KoRGbpXeTnzwxGfcfX+eQKP4/QRypeoECrFGYydgrIY/4VclEab+I/EpTh6Dv3coOT5hHcOL78B2bY1LT13iTFY9THxo
CQ7RNIU9MngJIVl03HYaUwIdelNRQuTAwrPpwU3pg18qTGa2TS3kfRSf3rRMo7SPDHxfjg9P6iIDqezeqBQ19m7udAxv+htjmY1i
+siM+IzwZgrX2Xg8dv9DK9yjmTB41rOwEcYIuXYh6WtbM+4q8pdhDyWvaXqmfSKNrMg8gQNOXiG/N11DBsM8paZiX1+9jpxbnFb4
0HdwsCcC+3cQxtfge8RkcYj0bLpr+mXkvGJyfQ62IoCoDcFZEpuYamqiaImykgp0JyCkryilWoV1J1TKcnK1Wu3wleHt/GZ2+zN8
ezWGnxZv4Y9DDyVr8K8wpSFThWuEq4ACDln/BfBGKyKadjHcc+H83bR7BvLjxDeL93O4nc9+Cf9PklZjIbgVSlL4AoeEljZu7/0F
fHj/FvTAlxMRrlBzJCGoiayGmJVdQ2vLgRa3LkhtFHxGrVJ451pF6TdHwbhSuhCSOVkqBdGeOJ2S0BHE751oOPdCGEvD1glTEURl
kcrykrMhP/2yFmviwaT7sBebNjlZFCLTN5xWQVuzEbaKojyc3S1uAMKEuP1hfje7uwnjM7+jRrC6jpp+0UohhcWorBUtlou7FBPx
5fUq7gdJOEDRq3ECr64T+Ob1U0EPHTro4tFy78WFRtWLar9bpDSZG1m57hcoOMPopfFR4SijfbzYLdBXbp9PjpfjVX/zGOkFlP4a
jSg6dDt1HI3+znFEYZh+VEJGfUFEYpFZlnc1dbfIhjiR9lsUp7T4WY2WGp+5bfLEakfsLqHHu9P0J/klDya3FNGnYlY1UfhuRu0O
n4exUxivLo8DP9GuCyTsxuNiphMt+0/1k5yzRkiVMS6KM86HxyFvHPLDtLgj7o6aQ138rJ5/qAUfOLYWottOWtH0V8mRWQIL+pmm
md8c/4gfAp4pjVTg3/ebfCQBxInfa8dI0A8Roe+XONlLffA3UEsDBBQAAAAIAAAAIQDEhHpxxAcAAAsRAAASAAAAY29sYWIvcmVz
b3VyY2VzLnB5jVfbjtvGGb7XU0w3FyRjiiKp02phGUjXi+ZgZ43dTRF0sSAocrQaL0+dGe5KEQQUqN2c7LYXTZA2PcRI0hZF4ThA
ktotjAB9FmujvEX/GZISpV055Y1GM//58P0zGxsb1+OzKIhdH/EBRn1yitFNEhEU925jjzPUp3EojxISRdhHZwOMA4WhJO0FxENv
7N1AoZsYGxsblQoJk5hy5PFRgtnyPyPlJCi2Bi4bBKRXkbIHnCeGFxAccZSfv3xwcGtn6OGEkzgqmG4zWEuOxOWCvaC+BX8LIjZi
GU1KAyAxMKUxXRIrNnRhtlwtEVP88xSzuRV72V9dHMcJjiqV3b1XfvLK66iLFGE026rVEhpzHJGhwWNW9aJqD5PbJDo2TuPAA6e9
OKwplRfQrSxYB7v7eVxRgKNjPmDIjXw0rApuEZSqR71WA3uhiwbY9TFlOvIG2DuBwNum3apaVtU2DZB4Mw04SVwwc+fAPWYojiBJ
hKFBDA64FKMo5pCrOMDVPgkgp9ebmSSWhsyo7P741Z3tg31wZVxB8CmFI04IyXd83HdBgXNqGk3DVLaQqkjmJCYRrz2P1ki4oqNm
vW02OnanoSPL7LTt+maz09i0Nxum2Wl2LE3PlHqe70CIkjiC5DNHGJrpisMwjmqLI8MjfRDb6JjtdrveskFsswnSLAuUbFp2q77Z
adlrxFL/hHAnjIMfUGDMCY3kJAB1VsNugHSrraPNZgM8aJqtjtm06uCDuVloS/ye4wUp45heVJDts2pvVAU1hI+qDdPgQxEj22p1
Os228MW27brdrrfrjbrVbHQgVM1CetxjkEOOHYoD7DLs+C53HY+dltVcRoQNQQTC64221RLOtEyzbbY37ZYFWjabttkCJZNKpQIZ
nNeGKpprS/aUjhh5C28hSDlUIfXkSkPVa8gnHt+S9pG+7EaDcZerGvw4ggf9qJvxShrxUZcwjH7qBimWfacq5x++/ew/30w/+9v0
0W+ePf719OFH5x8+mT18NH36wbPH97/7518VTXK/gKA1g7egI9Q3f4beABRhGkoZZhKUmBtiRHE/gJ6CHtnZvvkSSuJgFMUhcQPZ
XVCjHNa1Polg583dPeQy0YmGFJ8mIlbQBzlMbV+/cUMtQZYBbL4DJlCXjlRFGKJomiF+HdmsJSmGS48lJ4g7zGV4jjdwqZPoaL4h
IuPw0kYKcW01jsqSKAAPnC7sKqgkkU+OJTDxmINL3QJPDTZw7WZL1XRkSrozwgdZfgR+qQrtKZrwnnGK3XCRnbOBgAhvkEYnaKub
H4MNrq9a6OpVgB5tQbwwwMiMVSWjtkRQWFam0AXo5cS57VpRRBk9lI2os+dUzfbedquBnlcvFPOURmis9EYcM+gSEW4dQKFAVrHF
qZrph4MsaLCbOzXAw2ylakVz+PmIVGkc86I5oNekpP+nTXwxVrpIsKOa4JzvGoDhgAxGeOITqmZ/WPeApmAzHhJoqPhE/p2HSnKR
DC3VUl44HS0nKY/EvLN9WTNZOMDORcKwHLWlOC/LSVzGKtmCCi+kAaK0HJb2+2QoBRvZGl1BinCJ59now/R1OcchyCcRJDQ6xqoF
SF4yPO73GRaCBd8qkkiAgW0ZC6ZqCAdQEuZ6r4EhF3h1BYMWccnGfLeY8Go+16+IzOjF5O2OlT1hLlRGPyum7jiTPKkqk5IeadF4
ol1QJNsvvzyotLhNcBLiOOXdjil7ETodZhC7xM7cm4JARiYFbOlCP7YuJ19lyV0xjjGH3okjGNu8mnmliW7LHUNzx8YiZFVrUpOL
ibJejYzkRVT/4/3pew+mv//77NvfTT/+83d/uDN951fnD7+e3fvl9OOvlhp19cPBOmfN9VbMa8dcI3NdYC81v6+IKyIar1gxWWNx
Dq9QnRm8uj1ltSyUszWQe0HWMgTPLXgOCK9+oDpHbo6DAFrlSglw0bU13fCDQVGePX5/9vTp7Ju7s2/fzmB3+sWT2ZcPstm9JjbF
l1t0Rsnlg4JiD5NEZLB0A6GX45T4ZLgpTgLXwxJ4VsVJzMulriKcuruf3/8X/i3eAvryu0OmDRgvwMsyHokrxmXAdW31CrTkQRoF
JDpRl433MXeJmJpFHYJ2eEL40IdCL2EkAiUROA4H+uI5k4OimGpwsBIvCtMIChtqOpig//4LTR/9afbFB2ico/KkVhe740z3BC6M
fbizDkojp+R5geTQlfVLcFVWzv4IbrzhzpAIrbOv75z/+7dZAdXOP3nw/T/uTT/7cvbV51JnYdJcuZa9Nl+HO3k+eRNINczE0uDN
myB7CBavrzPccximp8TDho+hGX0ceSMHsLd4zEGSK6V6E1e08WQ+ok7wSEeqRP9F3YmBlb+TDKjekJWnLUQDRB4C45EA0vIEWZnC
F6Myffju7NO703t3AR6zCJ1/9Mn00R2Ax+m9J9/fvS+DAqLLwFMk8vzTX5z/5fM8eKqE6S19giSOa2vSVzh9CExHwnMFIgNzrfBA
Ry++uHTH0dFKLCYrdii7ryk50ZLww+I6dXTREjW//igAbnFKxftYvOkVLUMHh+MhV8WO4adhwtRCsA6JgHzyrq1lgnITXtrf34En
7O5rIlzNWhOCValAXhwngleB44gyVRwndEnkOPkkK+pJVJLKRkxc2U8PrSOQ/D9QSwMEFAAAAAgAAAAhAAHRPqMfEAAAGSgAABIA
AABjb2xhYi91YmlxdWl0aW4ucHmdWutz3NQV/75/xW3aGUlBFrt24iQe9oOJ12ZrJ3ZsJzNl2Wq00l1bjVbaSlrHJuMZ+kghkJBQ
SGgLFChvhiYwpBDSUGb6r5Rd25/6L/Scc69eu5sEmoFEex/nnnuev3OkQ4cOnQw8q8Wi2NrgEbN8h1ks5JYzEfjeDtviodt2ecja
QcjiTQ5TcS/0uYMbfMfyAp+zXsv9dc+NXZ+FPd84dOhQqeR2ukEYMyva8W03KLXDoMMcK+ax2+FMTia/dYZ/PwuUxLpNK9r03Fay
LNq0Jo9OiykxlJvs8NgCOlZy4K+iwBdLu1acp7ICP5NFIU+eol6rGwY2j6J0ZCcS+3s910k2nz1bnxOjz7rdtuulV3i6vmLO1eaX
Ztdrczp72u3Ow2R6+w3e6bilEpA0kBvD9SMexmpZB+GFKnKkmiaSM03NCHkUeFtc1WBtyP04alSa7HGmtCz7PPcdRdMEB1a3a8CV
uzEPgWwYxNx3txN+1FPLc7UlnS2vrNeXT6/pbGX25OLsQk1nazXkcEWunxUEsoFaGAahXmKP/mO7bRNvozPbc00r3IjEk+t3e7EO
Yg8jbnZD7rh27AZ+jmu4dM8G6wEzk+zW/S3Lc53lXkx7pbE5ZtdpiW0olF5oZzuWn/x57eQ6XGx5tb5QP10qnX2yfuZsfb1+mlWZ
cupMff7c4vrS+sLien19qXautrI2t16vnT63OLtYPzO3WFuor6zMnTmzulSfn11YPLNUm1tYXV9am/vF6fqZxdra+tJTS+eWVpdW
FxaU0lrtzNna6ZM1c+2pWaQ+OTXVOtIqt47bR6Yr0+UTR1v21OSRE+0TU8eOtu2pI0edqUmrfczhFm+Xj1Uqx5zy1PEp+2jFmW5X
TliWUlqvn6otn10HYpXj5XJpvr5UW4MfqpLocYKkaKAVKzpTMjEaIHYcSWVogIxwAD1OLM+Up4AgI9gUpYQ2rdC5AGaVDqSCTUdA
hxN8Cw2PhjxFK5VKDm8zP7igajNEXTh/6rgGTiW+a/RiWzPcKIBI0bFiNdl+IXRjbiJJVZgNuqukR14hFsR8O1ZxleH0Ot1IxVU6
c30HOKpOauwxpjyDfHLfDhzX36gqvbg9cTzl0t7k9nkwQghLnqeGQRDPkM/Lk6T5xEFob5bESBt93ZCigp3toDEz1WQ/AX1M6axS
gf+nNAZhL4kxydpMXYqGy6WPzaQKCC034mxtJ4p5p7btxqqy9/Lt/t9+t3f5hcGbn31/92r/yjcHl67+9/6V/dtfHzx/df/rV/Y+
+4xNsv7VG/1Lv4M7SQb9QLJs2D3HAuGa1pblQrj2eKKS8eedPDs3y/Cga7f3XvtYHNS/dvXgwxfYwsrZ/zz3m4NPvui/+HH/j1f6
377av3xVHLz3l98fvPbn/du3JQe+BcG6mudgg8emw7dcm5s4qZbFQtvqWi3Xc+OdBy7PliSb4H7K+hGFLgmpgw4DaedooSqO6ewo
aUGQlSog8jivVCaNaeUhomjDGUyIPxU8+/fX7CKet4tPa6fYxexQGqKzGAnx4ui5u1I8P2Wz4IpgbjvMs3o+7LDEnvMcMqSHl8KM
CS7hgcNAMvEh5IJLw99GIgFBHdwnUsHghKyqCp6iaEbU60A6AO+Af/CylUeqfP+763u3Xh+899zg7Q/673+xf+cDySt4N6gmTTto
+Xyb270YbUkzLrjxptBozrbFTnlz3J0mS0N4W0BxW20AcQggExNyqdKEjA7uXF0PezK5w8oqhDxBcZN7XfL3R9PEAIjBaWLzUUTb
nrURAcEG8EFBlLZNwBrTcUPxI+LcieDxsAo/FAgp5/kOARv8F7Qls6bWzPsfBM2IW6AlNVR++czaY3oqkclnjDL+Fx3+mZKO6rjh
FFms5e+oyFZi4dm98UyagVFi/OHOvFRnInQMXrjJUCSsf+23gxufg0Hvffah1HAXcAJBuCq76BhJzGoop0GrStPwggs8hKDBnMSW
iQu0yizAOS5YiNvqYcaJwOZABKOkdum4XFjHYItIZTjrYOpSuzvxZuBXc6HWiLrgaKrWKDf1lOtq8qALlzDRBaqjvveDIEr6Z6Pb
q6JV67moUvXgkmr2WxPwRR5SlcFcGn83BKeFGIIqWF7E0JAAJnZRrswFDBkrzISYadLkA+NIMQ7Nr0xNpuks5ICoQ67age+4MdlV
B5wEJJQkNGGd6fywAZ2zvB4nVKcmGwuZMgUAasht7nbjKM3x4uSIx9kUq4JVhcEF0BmZDTySwwg0ZmzhYWAwu+Bm+//4/eDe9cHd
S5BTMOLeujK4cUfaKG3lHuBf91lUSmiPo5JdZSPAGJFw0YCtzeyakk9Y01BaOzGPlCayiaSpjKEJOGL6CLc7lpyEAAhDWragF3pi
SgBKCApwyoiR4XLw63bP8wDbQCxQGuWJE9ZEu3lx+sgu2DrREpWK0tQyMbzz7sGnV/Zvfd7/9kaafkjJVBkx1IcEeTnMomMdZbrO
DPI7rBa4AZYjqliikWrEM3HpcV9NMTFNHpumCcFcNmcQkgIcoRmbfNtxwffAKXFDHvcW/U2B/LL/7pW9+zf7176GYNT/w6W9b28N
vnxj7847+etRxhkyM8J2XmA5kZpGDNuCVY8PYVEshSxHAEJN+qEseMAYhiqYH5HOQKr5YyWbkBy4Db6JyCWZB3FGCjwIsRaXGZ3z
8KzKGk1mI74NMcUMzlfnLS/iCTpY861utAkkIasDjLU5a3HwAEAEfpuHOGCwWYbYAFAP97zHI3BUDMy25aNvR1AZ+jGgC0iSUezG
vRi2xhFA4C03DPwOTBqpWxF8AmdSR7F+MS7n3EvNLv84EdAkGCd3StUkZkgpYkIqpbB9uKIYT+qhGi8QBwFbHUpmhw9L7Rs0hvAJ
VsG9JAYwI45RMFJmmKyvRIYC7cF2cq0gdDdcv6pkPQvTxq6HknhaVfwDQRZcwqvKKjoCn0M1mcJ3qg92jIRhYq4qeNdBcLG7xala
riZlcyY0UZPj3T2xLg6ySc2wIrMbRO42XHX4KLhGCIDStOIqFWfYT7DiXlRFy/WhOJK2ncvRRVWlNSNeX6yNw53MMERwQi9KOjiG
FLYqRZyzInF5WFpl1gXLjVmqLhnY0pCjs9zlhZC0lIwddLoeR5xcZflYUeB8tEjNBwuBLDwX0bTWmKg0M+ppupSHYGmiEs4G8/Hg
AMCZQ3cSebTQIBFYbPDmp4MXP+i/8uLBc5cHL31C+R1KrMHr7wy+vAHhfvDaV/1LV/fvfMOg9Bi8/PHe9T8oGScmSgGDLdwz3/BQ
C7LBJNAJbBlRMlOsZrGdKPJtm3djphb7OMONlXWhPJqE3Bfhthxg4FYkAD7kFpgR5dZOl9MPgDMYAEwzXf8DzAqcFn5RvwINE1xT
aUPhKsA8RzZgSJwLA23Xd6NNMmgYJpPezQQm9E2u/DBLyKtaTkC9jN01ddhYCdQdlosebD+TM00UfLeKnYeMnzElplCyKLgI1Imr
QaHIqI11Gnub/7/oop5tQ+kipDdGWFgoOS0ZpWBQ2NduHsIqK6u1ufpJrG8AyKZxT4fyoxdtUhpL8CEZ5Q41awotFAoHssVJs2RJ
WBlB8MokjMZChdhham8Bbx3Ld9vAkLjhKIKLAoxmqqRkIAHC6ALbyFkiizftf/vHwbvv7d2/1//8lcHN57//51cARAZvXu5f+ur7
f94cvPXu3hu38tqSh2BXqMyeYK5BDVdCiU9U2XGzXC7j/5REXcygCR/YEZJ8pMeKA/vvf9T//Fr//m/7d+/mjsIiCRMWsjqTkkH7
Iu61Qp6m+4hElcinGPhEyTUkO62ABbON6akS5dFmHGkW0V2eA1LPbno1BkltAvayQlUpkugIW6m9Jhy5jowfCTalNWBhsGI0oaft
aQVLcrBY0Z1WPGuHh34QdkwMP2j6VDVRK1TUh0lpBXNpa/uRQECKi7gSUEBA/lE8IIB+nnuC1xJb0wQhBDFOMAFninkDV2VoQCyV
t0+pSOeWbKT+nVtQxB5iZR6AFM5Mt+Ugh9iS4g4CynATbRzuwPEc4hhPOxdjiHRifo2hrnRTTwqF0cInr48RqxrXBW9SEEhfMCCn
OsuSIJrvrT8Nbn6z/69X+5c++BFFiDTkIhJtFpte0TjbL8LpIZdMJhtKrqkgxCXtleSZLUv6Hkozu39h/Yguxu8VjtKUzRXlMQX7
K2Rdk8Yxo6IMbc2aLNIGqY/6sLOojyMXTxmVilGZGiaaNVXEugY2b5sjRKnrCwEo27jR7cGO/NyI1ItVTVOTm3TcwSADiBbveINL
EUQjRxbTvAiI+IRnioNGgEWzAAuaBXU3OLnyBldEX4ToiN1CAom1mBKz5+oxE6pGOZK3FfwZdMC3HOXHNbvyr4vyx+VG5YmjKhEs
NypgSLjGxM6A0KGcmRgzVc5vnWw2RmL0Q81Y7puCfVa4sSUtJvMBnR1OXjCqslZ4IN/EneNaG34AhbINtNyIQNfQorGoH9eKMl6g
+8sfFSJXvn9VyCRZFQbnPZFKqgys9GK7MEb8pYO0Ow/jsoA5eP2rg9fvDNtvGl5ZdeT9aho6C28KgeJQoJSdd8sB3SCWpl1DcbuI
f2emirbeVlZrp2ZXF9mJo2X28+UnAeW6zq6S9pFEi1scgGJv5ddjqmSrtdklJbcMrr0y9yQD/NH/4q8F2GEHXWzI0At0gdAp+cSA
ZTfUsbwXFYRdMKRByaOSNsZwCBQ0fjSZODaNSeX2/f7z9xiyd/DqvyC+HNz4rn/vwwKT6PCWzlp4o2fdrppyJIjpLEd4ZhSQGuL1
WpW1xBN9emFA3kcshaPiMeHSIubwqaVJwQ1uvdR/+96wtaSsxUFHZ0G8ycOEQ+R2uMpN2IHVKUe0KccVzoG/YMcpm04G0hUAHugN
girm4aeG5m6Uy5XxkUxo/+W3+3+/js3Et64P3nn++7sv48hHL7FO52R9fgiNivfPZNr4cgylEYogTncc0kBS0fBtfPlbbLA+qrua
tlbxrGLfE+r8Gy/8kPah7B0i/IqywmC46TYGlGfbRj2b5TDdcC9JdHIgZPI446ed4aoOBJ2J9GudCXJgAwxDSRB67FoevsIlEqKN
GvXabcCECq7DDlycNJeK5SBtBUldgMCd/ypmfJGIn/iIV1W7BZsVL2vEFza+kAC9/4yGa/iWF7RgO64sNA+LGUKWYFR1o2oFedxb
XCjYEQUT1jGiiKJ1+fKp9EDCQ3WaznJfMgjiuW8ZBB1TJ6+JkjbQTiJELa8MuFzXs2yuCpUU30qdq63W5+u1OWw5VADrWiAzeMSG
v0VtCDpgV56DA1Kx9M6JXkO4bZb0dwjZmWYHqJimfKVO2EZnXmBb4iXU4VAUqthxx7TdqMzILBHQuxqqMpLluY+a0k8ukCCdRNB8
Qn6wkXuDP/ohhwRxXmG7dAtlqGnpBga4n5p/pyKZ1saSEaEhRyUXK5KN4/YJleU/PJD9lCGdZvw/SHHDNajoHxG13R+k2ATkehF/
2FvkHvwuyFxP3jzp8tLY9pP30kr/A1BLAwQUAAAACAAAACEAyp0o4RgEAACkCAAAIwAAAGNvbGFiL3JlcXVpcmVtZW50cy1saW51
eC1weTMxMS5sb2NrfVXZkuo4DH33vySVPfRDPsbYAgzeru0A6a8fyQ50M1NzXyi0RNZydMSPUVd+W5auHuuWcWtd4glklTYPcVma
+lA3jMeU3A0sKvq6qTt2VM5v6eLssrT1YSQ5qQQkDehPYuBXEjv0FhCSOil8pOmmusUAPRMXHiKkyrpguFbfECj2SN5aiduyHPDT
FgW3So8KDWRva/zSaRe44ZTcUE8oG0P/u2yzya2BCkJXUqzwZ1V3HhS3Al71fGor52Ml1rb7izm5IC5/c8oOb9smNBXU1C0VIUFg
xuixLCNmRQrw0QPI7DLXIwNl8ZnyfcfgCWJNyp5pLthxduIqVBANyQ3JSoN21KaB2tSxExaenNORNBPlcIr4gth7/oWKMxij6IWp
ntlZJXkkX0qRpNc4+9z2yzUWocUPlQARgOfHW/xWScuzI5ouRdsz5SkeRuuLtIf7qtsDxttlRNrZgE2x0lhiiDSltpgfSp4hRZo7
vXEFmXPtMCK7KnvlXUlnYld31OpIn06Y3HXF0BA0P1bvEITRdmY39VDR6TtNos3wNjzcVh/5CYrTgJrktUs5YE/QHH+pKmW1slDA
1TGjK8SeBpGUs3vuDTNOghbqlLHPjMfPLwV9aItnz2lKtAcDs5AeLtye9BTuAbNUV46DzWR2NWURBzLdlVQcAXfU/AVOxNL0aZT0
45P67XBoPuz2HpL4bZ/nD3tYbVIG/tfD2t30RR2ktr5tp1N6fZbLpQrfNgToy1i3bR7c2xi4lS9jg1/OH0+Wkf1EnvHj7pfZE3H8
5DtiS/5j1um9qxOywG41heqw21M71vO7kVYIvfvjnKefcPZ+VQgEe/to8PhjTs9/Nc75VOEux9XQkIkLURMA9kVvGOGBn8tqU2oe
e8Fj2fOWUfKusMDMcKNACIlEoDPaaPE9PHGrE20u7bRXWrtHLilbNU8nJFSpQiaCdqAwwRnMitjhplIB/tiTOoFVzxelkOyOK+IY
06Kl9JFeXpY5g9/jQQhOQMwklStZA1Rw57rsx8D8Vgm/KntyhJYcc+ulcbpcF4yxSY5oE5kyhvxFUeBeBchetIAvksjTaHMY6kvu
WZ9ZfWcTiZeq5Djsz6XvnfAmFDZuNFVDByvIXDvaRmxczwKSN8TyRk8AikKhB/ISD/moIeo+dBU88aRRqbTYaCk3ps1+mG3Yqihv
FG4eyQESLmYh5EOTZxWN4Z4uAIZgMSEOqAD+wmjczB6SUJMuyLjSYwCRNJVNlL7fmC4nlwW+SuU+NHcVFREvFk1Env5Ik09Cdggc
4ZwpEnd5yhqVCtET+nDE2GSqFE994biB/JqXBYHtC/2V4zuy9F1q2Ju+Bo282VNGlPADwZ2vQkeAeghk6JSP5BcO/3EByNgZiJp3
8rbH9+v7dZrYP1BLAQIUAxQAAAAIAAAAIQA6s6hp5QwAAHggAAAgAAAAAAAAAAAAAACAAQAAAABiYWNrZW5kL2FwcC9hZGFwdGVy
cy9wcm90ZW5peC5weVBLAQIUAxQAAAAIAAAAIQByFsKMyAMAAN4IAAAZAAAAAAAAAAAAAACAASMNAABiYWNrZW5kL2FwcC9zdHJ1
Y3R1cmVzLnB5UEsBAhQDFAAAAAgAAAAhAMSEenHEBwAACxEAABIAAAAAAAAAAAAAAIABIhEAAGNvbGFiL3Jlc291cmNlcy5weVBL
AQIUAxQAAAAIAAAAIQAB0T6jHxAAABkoAAASAAAAAAAAAAAAAACAARYZAABjb2xhYi91YmlxdWl0aW4ucHlQSwECFAMUAAAACAAA
ACEAyp0o4RgEAACkCAAAIwAAAAAAAAAAAAAAgAFlKQAAY29sYWIvcmVxdWlyZW1lbnRzLWxpbnV4LXB5MzExLmxvY2tQSwUGAAAA
AAUABQBmAQAAvi0AAAAA''')
if hashlib.sha256(payload).hexdigest() != 'a18d182f9bf1e57fd75cc0731b65cbd1d7aa87044834ea73b38ec88c789dfa3a':
    raise RuntimeError('笔记本内置文件校验失败')
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(support)
env = {k: os.environ[k] for k in ('HOME','PATH','LANG','TMPDIR','LD_LIBRARY_PATH','CUDA_HOME','CUDA_VISIBLE_DEVICES')
       if k in os.environ}
env.update(UV_PYTHON_INSTALL_DIR=str(root/'interpreters'), UV_CACHE_DIR=str(root/'uv-cache'),
           DS_BUILD_OPS='0', LAYERNORM_TYPE='torch', PROTENIX_ROOT_DIR=str(root/'cache'), PYTHONNOUSERSITE='1')

def call(args, timeout=1800):
    # Relay subprocess output to the notebook cell even when the kernel redirects its own stdout.
    import signal, threading
    from contextlib import suppress
    command = [str(s) for s in args]
    with subprocess.Popen(command, env=env, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors='replace', start_new_session=True) as process:
        def show_output():
            for line in process.stdout:
                print(line, end='', flush=True)
        reader = threading.Thread(target=show_output, daemon=True)
        reader.start()
        try:
            code = process.wait(timeout=timeout)
        except (KeyboardInterrupt, subprocess.TimeoutExpired):
            # SIGINT lets the Python inference driver cancel its own CLI process group.
            with suppress(ProcessLookupError):
                os.killpg(process.pid, signal.SIGINT)
            try:
                process.wait(timeout=8)
            except subprocess.TimeoutExpired:
                with suppress(ProcessLookupError):
                    os.killpg(process.pid, signal.SIGKILL)
            raise
        finally:
            reader.join(timeout=5)
        if code:
            raise subprocess.CalledProcessError(code, command)

bootstrap = root / 'bootstrap'
uv = bootstrap / 'bin/uv'
call([sys.executable, '-m', 'pip', '--isolated', 'install', '--no-deps', '--disable-pip-version-check',
      '--target', bootstrap, '--upgrade', '--index-url', 'https://pypi.org/simple', 'uv==0.9.18'])
tool = [uv, '--no-config']
call([*tool, 'python', 'install', '3.11.13'])
py = root / 'venv/bin/python'
if not py.is_file():
    call([*tool, 'venv', '--managed-python', '--python', '3.11.13', py.parents[1]])
pins = (support/'colab/requirements-linux-py311.lock').read_text().splitlines()
base = root / 'base.lock'
base.write_text('\n'.join(p for p in pins if not p.startswith(('deepspeed==', 'protenix=='))) + '\n')
install = [*tool, 'pip', 'install', '--python', py, '--no-deps', '--default-index', 'https://pypi.org/simple']
call([*install, '-r', base])
# DeepSpeed's source build can inspect torch and setuptools. Both were installed above.
wheel = ('https://files.pythonhosted.org/packages/a1/df/0f23f78428d4a5baaa6239ac1504994ee068b39a9facaff97f50cff547b7/'
         'protenix-2.0.0-py3-none-any.whl#sha256=be4584dda688e0518ce12909d8d51d0a0868e7daa2bbb95cbac17440c0ac309f')
call([*install, '--no-build-isolation-package', 'deepspeed', 'deepspeed==0.17.5', wheel])
call([*tool, 'pip', 'check', '--python', py])
runner = support / 'colab/ubiquitin.py'
call([py, runner, 'check-install', root])
install_ok = True

In [ ]:
# @title 3 · 下载并逐个校验 Mini 权重及四份公共缓存
if not install_ok:
    raise RuntimeError('请先完成第 2 单元')
assets_ok = prediction_ok = False
call([py, support/'colab/resources.py', root/'cache'], timeout=3600)
assets_ok = True

In [ ]:
# @title 4 · 校验内置运行包，安装独立 FastAPI 环境
if not assets_ok:
    raise RuntimeError('请先完成第 1–3 单元')
work = Path('/content/protenix-round6')
project = work/'project'
work.mkdir(exist_ok=True)
binary = base64.b64decode('''...''')
if hashlib.sha256(binary).hexdigest() != 'b240c2c4f47c0263184d0de06c374f5821d94696a093c0c339f9fd6ea3e2665f':
    raise RuntimeError('内置运行包摘要不符')
with zipfile.ZipFile(io.BytesIO(binary)) as z:
    z.extractall(project)
del binary
app_py = work/'backend-venv/bin/python'
if not app_py.is_file():
    call([*tool, 'venv', '--managed-python','--python','3.11.13',app_py.parents[1]])
call([*tool,'pip','install','--python',app_py,'--no-deps','--default-index','https://pypi.org/simple',
      '-r',project/'backend/requirements.lock'])
call([*tool,'pip','check','--python',app_py])
driver = project/'colab/round6.py'
call([app_py,driver,'--help'])
print('BACKEND READY · 单进程 · 仅使用 127.0.0.1')

In [ ]:
# @title 5 · 准备本机 Chromium，供第 6 单元操作 Mol*
ready = work/'browser-ready.json'
ready.unlink(missing_ok=True)
try:
    call([app_py,project/'colab/round6_browser_install.py',work], timeout=900)
    ready.write_text(json.dumps({'playwright':'1.62.1'}))
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print('BROWSER PARTIAL · 可继续第 6、7 单元。成功任务也可在 Mac 验证画布和下载。')

In [ ]:
# @title 6 · 发送一次聊天，在同一后端完成推理、Mol* 和下载验收
# The key lives only in this function and is sent to the backend on stdin.
def run_chat():
    from google.colab import userdata
    from contextlib import closing, suppress
    import signal, sqlite3, threading
    db = work/'data/app.sqlite3'
    committed = False
    if db.is_file():
        with closing(sqlite3.connect(db.as_uri()+'?mode=ro',uri=True)) as conn:
            committed = conn.execute('SELECT 1 FROM turns LIMIT 1').fetchone() is not None
    key = ''
    if not committed:
        try:
            key = userdata.get('DEEPSEEK_API_KEY')
        except Exception:
            raise RuntimeError('请在左栏 Secrets 添加 DEEPSEEK_API_KEY 并开启本笔记本的访问权限') from None
        if not isinstance(key,str) or not key.strip() or any(c in key for c in ('\n','\r')):
            raise RuntimeError('Secrets 中的 DeepSeek 凭证为空或带有换行')
    stage = 'view' if committed else 'run'
    with subprocess.Popen([str(app_py),str(driver),stage,str(work),str(root)], env=env, cwd=project,
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors='replace', start_new_session=True) as proc:
        proc.stdin.write(key.strip()+'\n')
        proc.stdin.close()
        del key
        def output():
            for line in proc.stdout:
                print(line,end='',flush=True)
        reader = threading.Thread(target=output,daemon=True)
        reader.start()
        try:
            code = proc.wait(timeout=2220)
        except (KeyboardInterrupt,subprocess.TimeoutExpired):
            with suppress(ProcessLookupError):
                os.killpg(proc.pid,signal.SIGINT)
            try:
                proc.wait(timeout=12)
            except subprocess.TimeoutExpired:
                with suppress(ProcessLookupError):
                    os.killpg(proc.pid,signal.SIGKILL)
            raise
        finally:
            reader.join(timeout=5)
        if code:
            raise RuntimeError('请查看上方的阶段或错误代码。可先运行第 7 单元保留已取得的证据。')
run_chat()

In [ ]:
# @title 7 · 下载本次聊天的证据 ZIP（也可下载部分结果）
from google.colab import files
latest = work/'latest.json'
if not latest.is_file():
    raise RuntimeError('尚无第六轮证据 ZIP，请反馈失败的单元和最后一条错误')
result = work/json.loads(latest.read_bytes())['file']
print('请放回项目 examples/outputs/：', result.name)
files.download(str(result))
call([app_py,driver,'verify',result])

完整成功会打印 **CHAT + GPU OK**、**STRUCTURE HTTP OK**、**MOLSTAR + DOWNLOAD OK** 和同一个任务 UUID。把第 7 单元的 **round6-protenix-….zip** 放回项目的 **examples/outputs/**。浏览器安装、画布或下载出错时，直接运行第 7 单元，保留该任务。也可在当前会话重跑第 5–7 单元，第 6 单元将只读复核已提交的任务。若 Colab 下载弹窗没有出现，左侧文件栏中可下载 /content/protenix-round6/ 下同名 ZIP。

反馈单元编号、最后一条脱敏错误和任务 UUID（若已生成）即可。GPU 故障可附 GPU 名和剩余显存 MiB。运行时回收后，已下载的成功任务可在 Mac 继续检查。[免费 Colab 资源说明](https://research.google.com/colaboratory/faq.html) · [Colab Secrets](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py)